# SFT + DPO Training & Assessment Pipeline

Full pipeline:
1. **SFT Training** (Phase 1: CPT + Phase 2: SFT)
2. **Full Test Set Assessment** with detailed report
3. **Generate DPO Preference Pairs** (auto-ranked by metrics)
4. **DPO Training** on top of SFT adapter
5. **Post-DPO Assessment** and compare SFT vs SFT+DPO
6. **Save everything** to Google Drive

**Requirements:** GPU runtime (T4 or better). Runtime > Change runtime type > GPU.

## 1. Setup & Install

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl rouge-score nltk sentence-transformers matplotlib pandas --no-build-isolation
!pip install -q -U flash-attn --no-build-isolation 2>/dev/null || echo 'flash-attn not available, will use eager attention'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# === Persistent paths on Google Drive ===
DRIVE_ROOT = '/content/drive/MyDrive/cocos2dx-finetune'
DRIVE_CHECKPOINTS = f'{DRIVE_ROOT}/checkpoints'
DRIVE_SFT_ADAPTER = f'{DRIVE_ROOT}/sft-adapter'
DRIVE_DPO_ADAPTER = f'{DRIVE_ROOT}/dpo-adapter'
DRIVE_REPORTS = f'{DRIVE_ROOT}/reports'
DRIVE_DPO_DATA = f'{DRIVE_ROOT}/dpo-preference-data'

for d in [DRIVE_CHECKPOINTS, DRIVE_SFT_ADAPTER, DRIVE_DPO_ADAPTER, DRIVE_REPORTS, DRIVE_DPO_DATA]:
    os.makedirs(d, exist_ok=True)

# Empty trash to free space
try:
    !find /content/drive/.Trash* -mindepth 1 -delete 2>/dev/null
    print('Drive trash emptied.')
except:
    pass

print(f'Drive storage ready at {DRIVE_ROOT}')

In [ ]:
import os

REPO = 'nmnhut-it/fine-tune-cocoos'
if not os.path.exists('fine-tune-cocoos'):
    !git clone https://github.com/{REPO}.git

os.chdir('fine-tune-cocoos')

## 2. Load & Clean Data

In [ ]:
from datasets import load_dataset, concatenate_datasets, Dataset
import json
import re

# Load all data files
train_ds = load_dataset('json', data_files='data/train.jsonl', split='train')
test_ds = load_dataset('json', data_files='data/test.jsonl', split='train')

# Load supplementary datasets
extra_files = [
    'data/conversational-qa.jsonl',
    'data/code-examples-qa.jsonl',
    'data/synthetic-qa.jsonl',
]
extra_datasets = []
for f in extra_files:
    if os.path.exists(f):
        ds = load_dataset('json', data_files=f, split='train')
        # Normalize columns to instruction/output
        if 'input' in ds.column_names:
            ds = ds.remove_columns([c for c in ds.column_names if c not in ('instruction', 'output')])
        extra_datasets.append(ds)
        print(f'  {f}: {len(ds)} examples')

if extra_datasets:
    for ds in extra_datasets:
        if set(ds.column_names) != set(train_ds.column_names):
            keep = [c for c in ds.column_names if c in ('instruction', 'output')]
            ds = ds.select_columns(keep)

print(f'\nBase Train: {len(train_ds)}, Test: {len(test_ds)}')
print(f'Extra datasets: {sum(len(d) for d in extra_datasets)} examples total')

In [ ]:
# === Data Cleaning ===

BROKEN_PREFIXES = [
    r"^Is it possible to how ",
    r"^I've been trying to how ",
    r"^So I need to what's ",
    r"^What do I need to know about What ",
    r"^What's the proper way to handle What ",
    r"^How exactly do you how ",
    r"^What's a clean way to how ",
]

def clean_instruction(text):
    """Fix broken template-generated instruction prefixes."""
    for pattern in BROKEN_PREFIXES:
        if re.match(pattern, text, re.IGNORECASE):
            text = re.sub(pattern, '', text, flags=re.IGNORECASE)
            text = text[0].upper() + text[1:] if text else text
    return text.strip()

def dedup_dataset(ds):
    """Remove near-duplicate examples based on output text."""
    seen_outputs = set()
    keep_indices = []
    for i, example in enumerate(ds):
        key = example['output'].strip().lower()[:200]
        if key not in seen_outputs:
            seen_outputs.add(key)
            keep_indices.append(i)
    return ds.select(keep_indices)

def clean_dataset(ds):
    ds = ds.map(lambda x: {'instruction': clean_instruction(x['instruction']), 'output': x['output']})
    before = len(ds)
    ds = dedup_dataset(ds)
    after = len(ds)
    if before != after:
        print(f'  Deduped: {before} -> {after} (removed {before - after})')
    return ds

print('Cleaning train set...')
train_ds = clean_dataset(train_ds)
print('Cleaning extra datasets...')
for i, ds in enumerate(extra_datasets):
    extra_datasets[i] = clean_dataset(ds)

# Combine all training data
all_train_cols = set(train_ds.column_names)
combined_extras = []
for ds in extra_datasets:
    cols_to_keep = [c for c in ds.column_names if c in all_train_cols]
    combined_extras.append(ds.select_columns(cols_to_keep))

if combined_extras:
    for i, ds in enumerate(combined_extras):
        for col in all_train_cols:
            if col not in ds.column_names:
                combined_extras[i] = ds.add_column(col, [''] * len(ds))
    full_train = concatenate_datasets([train_ds] + combined_extras)
else:
    full_train = train_ds

print('\nFinal dedup across combined training data...')
full_train = dedup_dataset(full_train)

print(f'\nFinal training set: {len(full_train)} examples')
print(f'Test set: {len(test_ds)} examples')

## 3. Model & Tokenizer Setup

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'Qwen/Qwen3-8B-Base'

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    tokenizer.add_special_tokens({'pad_token': '<|pad|>'})
tokenizer.padding_side = 'right'

attn_impl = 'flash_attention_2'
try:
    import flash_attn
except ImportError:
    attn_impl = 'eager'
    print('flash-attn not installed, using eager attention')

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=compute_dtype,
    attn_implementation=attn_impl,
)
model.resize_token_embeddings(len(tokenizer))
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    use_dora=True,
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

## 4. Tokenization (Two-Phase)

In [ ]:
ALPACA_TEMPLATE = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{output}"""

ALPACA_PROMPT = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
"""

MAX_LENGTH = 1024
eos_id = tokenizer.eos_token_id

def tokenize_full_text(example):
    """Phase 1 (CPT): full-text loss on everything."""
    full = ALPACA_TEMPLATE.format(instruction=example['instruction'], output=example['output'])
    tokens = tokenizer(full, truncation=True, max_length=MAX_LENGTH - 1, padding=False)
    tokens['input_ids'] = tokens['input_ids'] + [eos_id]
    tokens['attention_mask'] = tokens['attention_mask'] + [1]
    tokens['labels'] = list(tokens['input_ids'])
    return tokens

def tokenize_masked(example):
    """Phase 2 (SFT): loss only on response tokens."""
    prompt = ALPACA_PROMPT.format(instruction=example['instruction'])
    full = ALPACA_TEMPLATE.format(instruction=example['instruction'], output=example['output'])
    tokens = tokenizer(full, truncation=True, max_length=MAX_LENGTH - 1, padding=False)
    tokens['input_ids'] = tokens['input_ids'] + [eos_id]
    tokens['attention_mask'] = tokens['attention_mask'] + [1]
    prompt_tokens = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH)
    prompt_len = len(prompt_tokens['input_ids'])
    tokens['labels'] = [-100] * prompt_len + tokens['input_ids'][prompt_len:]
    return tokens

cpt_train = full_train.map(tokenize_full_text, remove_columns=full_train.column_names)
cpt_test = test_ds.map(tokenize_full_text, remove_columns=test_ds.column_names)
sft_train = full_train.map(tokenize_masked, remove_columns=full_train.column_names)
sft_test = test_ds.map(tokenize_masked, remove_columns=test_ds.column_names)

print(f'CPT data: {len(cpt_train)} train, {len(cpt_test)} test')
print(f'SFT data: {len(sft_train)} train, {len(sft_test)} test')

## 5. SFT Training (Phase 1: CPT + Phase 2: SFT)

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Any
from transformers import TrainingArguments, Trainer
import glob

@dataclass
class LabelPreservingCollator:
    tokenizer: Any
    max_length: int = 2048

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        pad_id = self.tokenizer.pad_token_id
        batch = {}
        for key in ('input_ids', 'attention_mask', 'labels'):
            seqs = [f[key] for f in features]
            pad_val = -100 if key == 'labels' else (0 if key == 'attention_mask' else pad_id)
            max_len = min(max(len(s) for s in seqs), self.max_length)
            padded = []
            for s in seqs:
                diff = max_len - len(s)
                padded.append(s + [pad_val] * diff if diff > 0 else s[:max_len])
            batch[key] = torch.tensor(padded, dtype=torch.long)
        return batch

use_bf16 = torch.cuda.is_bf16_supported()

def make_trainer(train_data, test_data, output_dir, num_epochs, lr):
    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type='cosine',
        bf16=use_bf16,
        fp16=not use_bf16,
        gradient_checkpointing=True,
        logging_steps=10,
        save_strategy='steps',
        save_steps=100,
        save_total_limit=3,
        report_to='none',
        dataloader_pin_memory=False,
        neftune_noise_alpha=5.0,
    )
    return Trainer(
        model=model, args=args,
        train_dataset=train_data, eval_dataset=test_data,
        data_collator=LabelPreservingCollator(tokenizer),
    )

In [ ]:
# === Phase 1: Continued Pre-Training (full-text loss) ===
print('=' * 60)
print('PHASE 1: Continued Pre-Training')
print('=' * 60)
cpt_dir = f'{DRIVE_CHECKPOINTS}/phase1-cpt'
os.makedirs(cpt_dir, exist_ok=True)

trainer = make_trainer(cpt_train, cpt_test, cpt_dir, num_epochs=5, lr=2e-4)
checkpoints = sorted(glob.glob(f'{cpt_dir}/checkpoint-*'))
resume = checkpoints[-1] if checkpoints else None
if resume:
    print(f'Resuming from {resume}')
trainer.train(resume_from_checkpoint=resume)

In [ ]:
# === Phase 2: Supervised Fine-Tuning (response-only loss) ===
print('=' * 60)
print('PHASE 2: Supervised Fine-Tuning')
print('=' * 60)
sft_dir = f'{DRIVE_CHECKPOINTS}/phase2-sft'
os.makedirs(sft_dir, exist_ok=True)

trainer = make_trainer(sft_train, sft_test, sft_dir, num_epochs=3, lr=5e-5)
checkpoints = sorted(glob.glob(f'{sft_dir}/checkpoint-*'))
resume = checkpoints[-1] if checkpoints else None
if resume:
    print(f'Resuming from {resume}')
trainer.train(resume_from_checkpoint=resume)

In [ ]:
# Save SFT adapter
model.save_pretrained(DRIVE_SFT_ADAPTER)
tokenizer.save_pretrained(DRIVE_SFT_ADAPTER)
print(f'SFT adapter saved to {DRIVE_SFT_ADAPTER}')

---
## 6. Post-SFT Assessment (Full Test Set)

Generate on every test example, compute metrics, save detailed report.

In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import json
import re
import time
from tqdm import tqdm

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

rouge_sc = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

def extract_identifiers(code):
    return set(re.findall(r'\b(?:cc|ccui|sp|ccs)\.[A-Za-z_.]+\b', code))

def compute_metrics(prediction, reference):
    ref_tokens = nltk.word_tokenize(reference.lower())
    pred_tokens = nltk.word_tokenize(prediction.lower())
    bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smooth)
    rouge = rouge_sc.score(reference, prediction)['rougeL'].fmeasure
    ref_ids = extract_identifiers(reference)
    pred_ids = extract_identifiers(prediction)
    if ref_ids:
        id_recall = len(ref_ids & pred_ids) / len(ref_ids)
        id_precision = len(ref_ids & pred_ids) / len(pred_ids) if pred_ids else 0
        id_f1 = 2 * id_precision * id_recall / (id_precision + id_recall) if (id_precision + id_recall) > 0 else 0
    else:
        id_recall = id_precision = id_f1 = None
    has_code = bool(re.search(r'```|\bfunction\b|\bconst\b|\bvar\b|\blet\b|=>|cc\.', prediction))
    return {'bleu': bleu, 'rouge_l': rouge, 'api_id_recall': id_recall, 'api_id_f1': id_f1, 'has_code': has_code}

def generate_response(mdl, instruction, max_new_tokens=512):
    prompt = ALPACA_PROMPT.format(instruction=instruction)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(mdl.device)
    with torch.no_grad():
        output = mdl.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, top_p=0.9, do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def run_full_test(mdl, test_dataset, label, save_path):
    """Run model on every test example, compute metrics, save results."""
    results = []
    if os.path.exists(save_path):
        with open(save_path) as f:
            results = json.load(f)
        print(f'Resuming {label} from {len(results)} completed examples')
    completed = {r['index'] for r in results}

    for i in tqdm(range(len(test_dataset)), desc=f'Testing ({label})'):
        if i in completed:
            continue
        ex = test_dataset[i]
        prediction = generate_response(mdl, ex['instruction'])
        metrics = compute_metrics(prediction, ex['output'])
        results.append({
            'index': i,
            'instruction': ex['instruction'],
            'reference': ex['output'],
            'prediction': prediction,
            **metrics,
        })
        if len(results) % 10 == 0:
            with open(save_path, 'w') as f:
                json.dump(results, f)

    with open(save_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'{label}: {len(results)} examples tested, saved to {save_path}')
    return results

print('Test functions ready.')

In [ ]:
# Run SFT test
model.eval()
sft_results = run_full_test(
    model, test_ds, 'SFT',
    f'{DRIVE_REPORTS}/sft_test_results.json'
)

In [ ]:
def avg(values):
    valid = [v for v in values if v is not None]
    return sum(valid) / len(valid) if valid else 0

def print_summary(results, label):
    print(f'\n{"=" * 50}')
    print(f'  {label} -- {len(results)} examples')
    print(f'{"=" * 50}')
    print(f'  BLEU-4:        {avg([r["bleu"] for r in results]):.4f}')
    print(f'  ROUGE-L:       {avg([r["rouge_l"] for r in results]):.4f}')
    print(f'  API ID Recall: {avg([r["api_id_recall"] for r in results]):.4f}')
    print(f'  API ID F1:     {avg([r["api_id_f1"] for r in results]):.4f}')
    print(f'  Has Code (%):  {avg([1 if r["has_code"] else 0 for r in results])*100:.1f}%')

print_summary(sft_results, 'POST-SFT')

---
## 7. Generate DPO Preference Pairs

For each training prompt, generate N completions, score against reference,
pick best as `chosen` and worst as `rejected`.

In [ ]:
import random
random.seed(42)

NUM_SAMPLES_PER_PROMPT = 5
DPO_SUBSET_SIZE = 1500
DPO_DATA_PATH = f'{DRIVE_DPO_DATA}/dpo_pairs.json'

dpo_prompt_indices = random.sample(range(len(full_train)), min(DPO_SUBSET_SIZE, len(full_train)))
print(f'Will generate DPO pairs for {len(dpo_prompt_indices)} prompts, {NUM_SAMPLES_PER_PROMPT} samples each')

In [ ]:
def score_completion(prediction, reference):
    """Combined score for ranking completions."""
    m = compute_metrics(prediction, reference)
    api_score = m['api_id_f1'] if m['api_id_f1'] is not None else 0.5
    return 0.4 * m['bleu'] + 0.3 * m['rouge_l'] + 0.3 * api_score

def generate_n_samples(mdl, instruction, n=5, max_new_tokens=512):
    """Generate N diverse completions with higher temperature."""
    prompt = ALPACA_PROMPT.format(instruction=instruction)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(mdl.device)
    samples = []
    for _ in range(n):
        with torch.no_grad():
            output = mdl.generate(
                **inputs, max_new_tokens=max_new_tokens,
                temperature=0.9, top_p=0.95, do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        text = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        samples.append(text)
    return samples

# Resume support
dpo_pairs = []
if os.path.exists(DPO_DATA_PATH):
    with open(DPO_DATA_PATH) as f:
        dpo_pairs = json.load(f)
    print(f'Resuming DPO generation from {len(dpo_pairs)} pairs')
completed_dpo = {p['prompt_index'] for p in dpo_pairs}

model.eval()
for idx in tqdm(dpo_prompt_indices, desc='Generating DPO pairs'):
    if idx in completed_dpo:
        continue
    ex = full_train[idx]
    instruction = ex['instruction']
    reference = ex['output']

    samples = generate_n_samples(model, instruction, n=NUM_SAMPLES_PER_PROMPT)
    scored = [(s, score_completion(s, reference)) for s in samples]
    scored.sort(key=lambda x: x[1], reverse=True)

    best = scored[0]
    worst = scored[-1]

    # Only keep pairs with meaningful score difference
    if best[1] - worst[1] > 0.05:
        prompt_text = ALPACA_PROMPT.format(instruction=instruction)
        dpo_pairs.append({
            'prompt_index': idx,
            'prompt': prompt_text,
            'chosen': best[0],
            'rejected': worst[0],
            'chosen_score': best[1],
            'rejected_score': worst[1],
        })

    if len(dpo_pairs) % 50 == 0:
        with open(DPO_DATA_PATH, 'w') as f:
            json.dump(dpo_pairs, f)

with open(DPO_DATA_PATH, 'w') as f:
    json.dump(dpo_pairs, f, indent=2)

print(f'\nGenerated {len(dpo_pairs)} DPO preference pairs')
print(f'Avg chosen score: {avg([p["chosen_score"] for p in dpo_pairs]):.4f}')
print(f'Avg rejected score: {avg([p["rejected_score"] for p in dpo_pairs]):.4f}')
print(f'Avg score gap: {avg([p["chosen_score"] - p["rejected_score"] for p in dpo_pairs]):.4f}')

---
## 8. DPO Training

In [ ]:
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

dpo_dataset = Dataset.from_dict({
    'prompt': [p['prompt'] for p in dpo_pairs],
    'chosen': [p['chosen'] for p in dpo_pairs],
    'rejected': [p['rejected'] for p in dpo_pairs],
})

dpo_split = dpo_dataset.train_test_split(test_size=0.1, seed=42)
print(f'DPO train: {len(dpo_split["train"])}, held-out: {len(dpo_split["test"])}')

In [ ]:
dpo_dir = f'{DRIVE_CHECKPOINTS}/phase3-dpo'
os.makedirs(dpo_dir, exist_ok=True)

dpo_config = DPOConfig(
    output_dir=dpo_dir,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    bf16=use_bf16,
    fp16=not use_bf16,
    gradient_checkpointing=True,
    logging_steps=10,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=3,
    report_to='none',
    max_length=1024,
    max_prompt_length=512,
    beta=0.1,
    loss_type='sigmoid',
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_split['train'],
    eval_dataset=dpo_split['test'],
    processing_class=tokenizer,
)

checkpoints = sorted(glob.glob(f'{dpo_dir}/checkpoint-*'))
resume = checkpoints[-1] if checkpoints else None
if resume:
    print(f'Resuming DPO from {resume}')

print('Starting DPO training...')
dpo_trainer.train(resume_from_checkpoint=resume)

In [ ]:
model.save_pretrained(DRIVE_DPO_ADAPTER)
tokenizer.save_pretrained(DRIVE_DPO_ADAPTER)
print(f'DPO adapter saved to {DRIVE_DPO_ADAPTER}')

---
## 9. Post-DPO Assessment (Full Test Set)

In [ ]:
model.eval()
dpo_results = run_full_test(
    model, test_ds, 'DPO',
    f'{DRIVE_REPORTS}/dpo_test_results.json'
)

---
## 10. Comparison Report: SFT vs SFT+DPO

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Reload results if needed (after restart)
if not sft_results:
    with open(f'{DRIVE_REPORTS}/sft_test_results.json') as f:
        sft_results = json.load(f)
if not dpo_results:
    with open(f'{DRIVE_REPORTS}/dpo_test_results.json') as f:
        dpo_results = json.load(f)

def summarize_metrics(results):
    return {
        'BLEU-4': avg([r['bleu'] for r in results]),
        'ROUGE-L': avg([r['rouge_l'] for r in results]),
        'API ID Recall': avg([r['api_id_recall'] for r in results]),
        'API ID F1': avg([r['api_id_f1'] for r in results]),
        'Has Code %': avg([1 if r['has_code'] else 0 for r in results]),
    }

sft_summary = summarize_metrics(sft_results)
dpo_summary = summarize_metrics(dpo_results)

print(f'{"Metric":<20} {"SFT":>10} {"SFT+DPO":>10} {"Delta":>10}')
print('-' * 52)
for key in sft_summary:
    s = sft_summary[key]
    d = dpo_summary[key]
    delta = d - s
    arrow = '+' if delta > 0 else ''
    print(f'{key:<20} {s:>10.4f} {d:>10.4f} {arrow}{delta:>9.4f}')

In [ ]:
# === Per-Example Comparison Table ===
sft_by_idx = {r['index']: r for r in sft_results}
dpo_by_idx = {r['index']: r for r in dpo_results}

rows = []
for idx in sorted(sft_by_idx.keys()):
    s = sft_by_idx[idx]
    d = dpo_by_idx.get(idx)
    if d is None:
        continue
    rows.append({
        'idx': idx,
        'instruction': s['instruction'][:80] + '...',
        'sft_bleu': s['bleu'],
        'dpo_bleu': d['bleu'],
        'sft_rouge': s['rouge_l'],
        'dpo_rouge': d['rouge_l'],
        'sft_api_f1': s['api_id_f1'] or 0,
        'dpo_api_f1': d['api_id_f1'] or 0,
        'winner': 'DPO' if d['rouge_l'] > s['rouge_l'] else 'SFT',
    })

df = pd.DataFrame(rows)
dpo_wins = (df['winner'] == 'DPO').mean() * 100
sft_wins = (df['winner'] == 'SFT').mean() * 100
print(f'\nWin rate -- DPO: {dpo_wins:.1f}%  SFT: {sft_wins:.1f}%')
df.to_csv(f'{DRIVE_REPORTS}/per_example_comparison.csv', index=False)
print(f'Saved to {DRIVE_REPORTS}/per_example_comparison.csv')
df.head(20)

In [ ]:
# === Comparison Chart ===
metrics_names = list(sft_summary.keys())
sft_vals = [sft_summary[k] for k in metrics_names]
dpo_vals = [dpo_summary[k] for k in metrics_names]

x = range(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar([i - width/2 for i in x], sft_vals, width, label='SFT', color='#4CAF50')
bars2 = ax.bar([i + width/2 for i in x], dpo_vals, width, label='SFT + DPO', color='#FF9800')

ax.set_ylabel('Score')
ax.set_title('SFT vs SFT + DPO -- Full Test Set Comparison')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.set_ylim(0, 1)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
               xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
chart_path = f'{DRIVE_REPORTS}/sft_vs_dpo_chart.png'
plt.savefig(chart_path, dpi=150)
plt.show()
print(f'Chart saved to {chart_path}')

In [ ]:
# === Side-by-Side Examples ===
from IPython.display import display, HTML
import html

def show_sft_vs_dpo(idx):
    s = sft_by_idx[idx]
    d = dpo_by_idx[idx]
    display(HTML(f"""
    <div style='border:1px solid #ccc; padding:12px; margin:8px 0; border-radius:8px'>
    <h3>Test Example {idx}</h3>
    <p><b>Instruction:</b> {html.escape(s['instruction'][:300])}</p>
    <table style='width:100%; border-collapse:collapse'>
    <tr>
      <th style='width:50%; border:1px solid #ddd; padding:8px; background:#e8f5e9'>SFT (BLEU={s['bleu']:.3f}, ROUGE={s['rouge_l']:.3f})</th>
      <th style='width:50%; border:1px solid #ddd; padding:8px; background:#fff3e0'>SFT+DPO (BLEU={d['bleu']:.3f}, ROUGE={d['rouge_l']:.3f})</th>
    </tr>
    <tr>
      <td style='border:1px solid #ddd; padding:8px; vertical-align:top'><pre style='white-space:pre-wrap; font-size:11px'>{html.escape(s['prediction'][:800])}</pre></td>
      <td style='border:1px solid #ddd; padding:8px; vertical-align:top'><pre style='white-space:pre-wrap; font-size:11px'>{html.escape(d['prediction'][:800])}</pre></td>
    </tr>
    <tr><td colspan='2' style='border:1px solid #ddd; padding:8px; background:#e3f2fd'>
      <b>Reference:</b><pre style='white-space:pre-wrap; font-size:11px'>{html.escape(s['reference'][:500])}</pre>
    </td></tr>
    </table></div>
    """))

df_sorted = df.copy()
df_sorted['rouge_delta'] = df_sorted['dpo_rouge'] - df_sorted['sft_rouge']

print('=== Top 5 DPO Improvements ===')
for idx in df_sorted.nlargest(5, 'rouge_delta')['idx'].tolist():
    show_sft_vs_dpo(idx)

print('\n=== Top 5 DPO Regressions ===')
for idx in df_sorted.nsmallest(5, 'rouge_delta')['idx'].tolist():
    show_sft_vs_dpo(idx)

---
## 11. Save Full Report

In [ ]:
report = {
    'model': MODEL_ID,
    'num_test_examples': len(test_ds),
    'num_train_examples': len(full_train),
    'num_dpo_pairs': len(dpo_pairs),
    'sft_metrics': sft_summary,
    'dpo_metrics': dpo_summary,
    'delta': {k: dpo_summary[k] - sft_summary[k] for k in sft_summary},
    'dpo_win_rate': dpo_wins / 100,
    'sft_win_rate': sft_wins / 100,
    'training_config': {
        'sft_phase1_epochs': 5,
        'sft_phase1_lr': 2e-4,
        'sft_phase2_epochs': 3,
        'sft_phase2_lr': 5e-5,
        'dpo_epochs': 2,
        'dpo_lr': 5e-6,
        'dpo_beta': 0.1,
        'lora_r': 64,
        'lora_alpha': 128,
    },
}

report_path = f'{DRIVE_REPORTS}/full_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))
print(f'\nFull report saved to {report_path}')
print(f'\nAll outputs in {DRIVE_REPORTS}/:')
for fname in os.listdir(DRIVE_REPORTS):
    size = os.path.getsize(f'{DRIVE_REPORTS}/{fname}') / 1024
    print(f'  {fname} ({size:.1f} KB)')

---
## 12. (Optional) Push to Hugging Face Hub

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()
# model.push_to_hub('your-username/cocos2dx-coder-dpo')
# tokenizer.push_to_hub('your-username/cocos2dx-coder-dpo')